# OpenVINO Physical AI APIs enabling deployment of optimized model

### Minimal code to run a Pi0.5 model on an SO101 robot

<img src="media/2. studio+OV.png" width="800">

## 1) Train a model using the Physical AI Studio or LeRobot
* Use Pi0.5 policy supported in Physical AI Studio or LeRobot to train a model.<br>
* The output of this stage should be a trained Pi0.5 model<br>
<font size="2">[Open Notebook](002_Using_Physical_AI_Studio.ipynb)

## 2) Install OpenVINO Physical AI on your target platform

Clone the OpenVINO Physical AI repo and install.

In [ ]:
from pathlib import Path

requirements_file = Path("requirements.txt")
if not requirements_file.exists():
    requirements_file = Path("notebooks/requirements.txt")

if not Path("physicalai").exists():
    !git clone https://github.com/openvinotoolkit/physicalai.git

%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu -r {requirements_file}
%pip install -q -e physicalai

Import libraries, and set up a working directory

In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import openvino as ov
from huggingface_hub import hf_hub_download, snapshot_download
from physicalai.inference import InferenceModel

WORKSPACE = Path.cwd().resolve()
RUNTIME_ROOT = WORKSPACE / "physicalai"
ROOT = RUNTIME_ROOT
os.chdir(ROOT)

Install extras, mainly for hardware acceleration (e.g. for SO101 or specific camera types)

In [ ]:
INSTALL_HARDWARE_DEPS = "True"
USE_SHARED_CAMERA = "True"

if INSTALL_HARDWARE_DEPS:
    import subprocess
    import sys

    hardware_extras = "transport,so101" if USE_SHARED_CAMERA else "so101"
    runtime_with_hardware_extras = str(WORKSPACE / "physicalai") + f"[{hardware_extras}]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", runtime_with_hardware_extras])
    print(f"[DONE] Installed PhysicalAI hardware extras: {hardware_extras}")
else:
    print("[SKIP] Hardware extras are not installed. Set INSTALL_HARDWARE_DEPS = True if this environment needs them.")


## 3) Discover and connect to Robot and Cameras

Discover cameras.

In [ ]:
# Discover available cameras — use the device IDs from this output in the config below
from physicalai.capture import discover_all

for driver, devices in discover_all().items():
    if not devices:
        continue
    print(f"\n[{driver}]")
    for dev in devices:
        print(f"  {dev.device_id}  —  {dev.name}")


To set up cameras<br>
* Use the output of previous cell in the next cell<br>
* Use the same names of camera as set in the dataset<br>
<img src="media/detect-cameras.png" alt="Description" width="800" height="200">


In [ ]:
# Cameras — use the same names and setup as used for the dataset collection
# ============================================================
# 🔴 USER PARAMETERS — Edit these before running  🔧
# ============================================================
CAMERAS = [
    (<camera-name>, "uvc", <camera description>),
    (<camera-name>, "uvc", <camera description>),
]

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
CAMERA_FPS = 30

# Runtime
FPS = 30
DURATION_S = 60.0

Define robot and get the robot ID from Physical AI Studio<br><br>
<img src="media/robot-ID.png" alt="Description" width="800" height="267">



In [ ]:
# ============================================================
# 🔴 USER PARAMETERS — Edit these before running  🔧
# ============================================================
# Robot
SO101_PORT = "/dev/ttyACM0"

# ============================================================
# 🔴 USER PARAMETERS — Edit these before running  🔧
# ============================================================
# Calibration file location depends on how you collected data:
#   Physical AI Studio: ~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json
#   LeRobot:            ~/.cache/calibration/so101_follower.json

#SO101_CALIBRATION = "~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json"
SO101_CALIBRATION = "/home/intel/.cache/physicalai/robots/7549dd0c-a292-41c4-a8c0-712aae14c55f/calibrations/b28a2ed2-66db-4f61-848e-b65c88827d48.json"

Connect cameras and robot together. This is an important step as it sets up an environment for robot and camera together.
* Make sure Physical AI Studio application is closed and no other application is using the cameras or robot

In [ ]:
from physicalai.capture import SharedCamera
from physicalai.robot import SO101

# Cameras
cameras = {}
for name, driver, device_id in CAMERAS:
    kwargs = {"serial_number": device_id} if driver == "realsense" else {"device": device_id}
    cam = SharedCamera(driver, **kwargs, width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS)
    cam.connect()
    cameras[name] = cam
    print(f"Camera '{name}' connected: {cam.actual_width}x{cam.actual_height} @ {cam.actual_fps}fps")

# Robot
robot = SO101(port=SO101_PORT, calibration=SO101_CALIBRATION, role="follower")
robot.connect()
print(f"Robot connected on {SO101_PORT}")

## 4) Load the model

In [ ]:
# ============================================================
# 🔴 USER PARAMETERS — Edit these before running  🔧
# ============================================================
EXPORT_DIR = "exports/pi05-pick-place-purple-cube"
DEVICE = "GPU"  # OpenVINO can run on "GPU", "CPU", "NPU"
TASK = "pick up the box"

<br>Make sure to copy all the model files from the studio to the $EXPORT_DIR
<br>Then, load the model to the target device
<br><br>
<img src="media/download-model.png" alt="Description" width="800" height="200">

In [ ]:
import openvino_tokenizers 
from physicalai.inference import InferenceModel

policy = InferenceModel.load(EXPORT_DIR, device=DEVICE)
print(f"Model loaded on {DEVICE}")

## 5) Run the model (policy) on the robot

Use OpenVINO Physical AI to perform Inference.

In [ ]:
from physicalai.runtime import ActionQueue, PolicyRuntime, SyncExecution

runtime = PolicyRuntime(
    robot=robot,
    model=policy,
    execution=SyncExecution(fps=FPS, request_threshold=0.5),
    fps=FPS,
    cameras=cameras,
    action_queue=ActionQueue(),
    task=TASK,
)

with runtime:
    print(f"Running policy at {FPS} fps for {DURATION_S}s — task: {TASK!r}")
    stats = runtime.run(duration_s=DURATION_S)

print(f"\nDone — {stats.steps} steps, {stats.inference_count} inferences, {stats.total_holds} holds")

## 6) Disconnect

In [ ]:
for name, cam in cameras.items():
    cam.disconnect()
    print(f"Camera '{name}' disconnected")

robot.disconnect()
print("Robot disconnected")